In [370]:
# make data dictionary
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DESCRIP_FILE = PROJECT_ROOT/"data"/"descriptions.txt"

data_url = "https://cmustatistics.github.io/data-repository/data/vietnam-health.csv"
df = pd.read_csv(data_url)

data_descriptions = []
with open(DESCRIP_FILE, 'r', encoding='utf-8') as file:
    for line in file:
        data_descriptions.append(line.strip())

data_values = []
for col in df.columns:
    if df[col].dtype == 'object':
        data_values.append(df[col].unique())
    if (df[col].dtype == 'int64') | (df[col].dtype == 'float64'):
        data_values.append(f"{df[col].min()} - {df[col].max()}")

data_dict = pd.DataFrame({
    'Variable Name':df.columns,
    'Data Type':df.dtypes,
    'Data Description': data_descriptions,
    'Values/Ranges':data_values
    }).reset_index(drop=True)

data_dict

,Variable Name,Data Type,Data Description,Values/Ranges
0,id,object,A unique ID for each survey respondent,"[m00001, m00002, m00003, m00004, m00005, m0000..."
1,date,int64,"Probably the date of the interview, in YYYYMMD...",20160110 - 20169828
2,Age,int64,Age of survey respondent (years),13 - 83
3,Sex,object,Sex of survey respondent,"[female, male]"
4,Jobstt,object,"Job status: stable, unstable, student, retired...","[student, unstable, stable, other, housewife, ..."
5,MaritalStt,object,"Marital status: married, unmarried, other","[unmarried, married, other]"
6,Edu,object,Maximum educational attainment (PostGrad = pos...,"[Grad, Second, Hi, PostGrad]"
7,height,float64,Height (cm),140.0 - 188.0
8,weight,float64,Weight (kg),32.0 - 105.0
9,place,object,Place the survey was conducted,"[hanoi, quangninh, hanam, haiduong, bacninh, n..."


In [371]:
#write dictionary to md file
markdown_data_dict = data_dict.to_markdown(index=False)
with open('../data/data_dictionary.md', 'w') as file:
    file.write(markdown_data_dict)

Understand and clean data

In [372]:
df.isna().sum()[df.isna().sum()>0]

Series([], dtype: int64)

In [373]:
df.duplicated().sum()

np.int64(0)

In [374]:
df.columns

Index(['id', 'date', 'Age', 'Sex', 'Jobstt', 'MaritalStt', 'Edu', 'height',
       'weight', 'place', 'BMI', 'HealthIns', 'RecExam', 'RecPerExam',
       'ReaExam', 'Wsttime', 'Wstmon', 'DiscDisease', 'Lessbelqual', 'NotImp',
       'HthyPriority', 'ComSubsidy', 'Habit', 'FlwHealth', 'PerTrmt',
       'AcqTrmt', 'StabHthStt', 'MedCabinet', 'Tooluseskills', 'ExpCare',
       'ExamTools', 'UseMon', 'Tangibles', 'Reliability', 'Respon',
       'Assurance', 'Empathy', 'StChoise', 'CHPerc', 'SuitFreq', 'AffCost',
       'UseIT', 'AfterIT', 'SuffInfo', 'AttractInfo', 'ImpressInfo',
       'PopularInfo', 'SuitExer', 'EvalExer', 'Age_gr'],
      dtype='object')

Rename variables to be more interpretable

In [375]:
df["Edu"] = df["Edu"].replace({'Second':'High', 'Hi':'Middle'})
print(df["Edu"].value_counts(sort=False))

Edu
Grad        1383
High         142
Middle       416
PostGrad     127
Name: count, dtype: int64


In [ ]:
df["RecExam"] = df["RecExam"].replace({'less12':'<12mo', 'b1224':'12-24mo',
                                       'g24':'>24mo', 'unknow':'unknown'})
print(df["RecExam"].value_counts(sort=False))

RecExam
<12mo      1373
unknown     370
12-24mo     200
>24mo       125
Name: count, dtype: int64


In [ ]:
df["RecPerExam"] = df["RecPerExam"].replace({'less12':'<12mo', 'b1224':'12-24mo',
                                       'g24':'>24mo', 'unknow':'unknown'})
print(df["RecPerExam"].value_counts(sort=False))

RecPerExam
<12mo      1059
unknown     516
>24mo       275
12-24mo     218
Name: count, dtype: int64


In [ ]:
df["ReaExam"] = df["ReaExam"].replace({'noti.disease':'disease_concern',
                                       'adv.sig':'symptoms'})
print(df["ReaExam"].value_counts(sort=False))

ReaExam
request            726
volunteer          514
disease_concern     98
symptoms           730
Name: count, dtype: int64


In [ ]:
df["EvalExer"] = df["EvalExer"].replace({'quitesuff':'disease_concern',
                                       'trivial':'symptoms',
                                       'little':'symptoms',
                                       'verysuff':'symptoms'})
print(df["ReaExam"].value_counts(sort=False))

ReaExam
request            726
volunteer          514
disease_concern     98
symptoms           730
Name: count, dtype: int64


In [ ]:
df['date'] = df['date'].astype(str)
print(df['date'].dtype)
df['date'].head()
for date in df['date']:
    if (("-" not in date) & (date != 'invalid')):
        year = date[0:4]
        month = date[4:6]
        day = date[6:]
        if ((int(year) > 2017) | (int(month) > 12) | (int(day) > 31)):
            (df.loc[df['date'] == date]).astype(str) = 'invalid'
        else:
            (df.loc[df['date'] == date]).astype(str) = f'{str(year)}-{str(month)}-{str(day)}'
#df['date'].unique()

C:\Users\emmag\AppData\Local\Temp\ipykernel_34000\3356906930.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2016-09-19' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df['date'] == date] = f'{str(year)}-{str(month)}-{str(day)}'
C:\Users\emmag\AppData\Local\Temp\ipykernel_34000\3356906930.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2016-09-19' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['date'] == date] = f'{str(year)}-{str(month)}-{str(day)}'
C:\Users\emmag\AppData\Local\Temp\ipykernel_34000\3356906930.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2016-09-19' has dtype incompatible with float64, please explicitly cast 

ValueError: invalid literal for int() with base 10: '9-19'

In [ ]:
df.to_csv('../data/clean/clean_vietnam-health.csv')